In [1]:
from transformers import pipeline

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [2]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
text = "I love learning machine learning. It is amazing!"

result = sentiment_pipeline(text)

print(result)

[{'label': 'positive', 'score': 0.9895942211151123}]


In [4]:
text = "I studied for months, but I failed my exam. I feel terrible."

result = sentiment_pipeline(text)

print(result)

[{'label': 'negative', 'score': 0.9346500039100647}]


In [5]:
text = "The meeting starts at 10 AM tomorrow."

result = sentiment_pipeline(text)

print(result)

[{'label': 'neutral', 'score': 0.9208857417106628}]


In [6]:
import pandas as pd

df = pd.read_csv("../data/processed_data.csv")

print(df.shape)
print(df["full_text"].head())

(10, 8)
0    Model Selection Technique for Causal Inference...
1    Please Please Please do not get a Data Science...
2     Incoming Google DS-R interview — what to expect?
3                 Data Science Online course [removed]
4                            Maximum Likelihood Monday
Name: full_text, dtype: str


In [7]:
text = df["full_text"].iloc[0]

result = sentiment_pipeline(
    text,
    truncation=True,
    max_length=512
)

print("Prediction:", result)

Prediction: [{'label': 'neutral', 'score': 0.7642422318458557}]


In [8]:
def predict_sentiment(text):
    result = sentiment_pipeline(
        text,
        truncation=True,
        max_length=512
    )

    return result[0]

In [9]:
print(predict_sentiment("I love learning machine learning!"))

{'label': 'positive', 'score': 0.9891586899757385}


In [10]:
predictions = df["full_text"].apply(predict_sentiment)

In [11]:
df["sentiment"] = predictions.apply(lambda x: x["label"])
df["sentiment_score"] = predictions.apply(lambda x: x["score"])

In [12]:
print(df[["title", "sentiment", "sentiment_score"]])

                                               title sentiment  \
0     Model Selection Technique for Causal Inference   neutral   
1  Please Please Please do not get a Data Science...  negative   
2   Incoming Google DS-R interview — what to expect?   neutral   
3                         Data Science Online course   neutral   
4                          Maximum Likelihood Monday   neutral   
5   Incoming Google DS-R interview — what to expect?   neutral   
6   Why Is Ridge Giving Large Negative Coefficients?  negative   
7  A Library for Exploring Predictive Multiplicit...   neutral   
8  Backtesting an ensemble model's picks against ...   neutral   
9  Weekly Entering & Transitioning - Thread 05 Oc...   neutral   

   sentiment_score  
0         0.764242  
1         0.649543  
2         0.918867  
3         0.922719  
4         0.707494  
5         0.943661  
6         0.527360  
7         0.927111  
8         0.920560  
9         0.826192  


In [13]:
df.to_csv("../data/sentiment_results.csv", index=False)

In [14]:
import os

print(os.path.exists("../data/sentiment_results.csv"))

True


In [15]:
import sys
sys.path.append("..")

from ml.sentiment import load_sentiment_pipeline, predict_sentiment

In [16]:
sentiment_pipeline = load_sentiment_pipeline()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
text = "I love learning machine learning. It is amazing!"

result = predict_sentiment(text, sentiment_pipeline)

print(result)

{'label': 'positive', 'score': 0.9895942211151123}


In [18]:
import pandas as pd

In [19]:
df = pd.read_csv("../data/processed_data.csv")

In [20]:
df["predictions"] = df["full_text"].apply(predict_sentiment, args=(sentiment_pipeline,))

In [21]:
df["sentiment"] = df["predictions"].apply(lambda x: x["label"])

In [22]:
df["sentiment_score"] = df["predictions"].apply(lambda x: x["score"])

In [23]:
print(df[["title","sentiment","sentiment_score"]])

                                               title sentiment  \
0     Model Selection Technique for Causal Inference   neutral   
1  Please Please Please do not get a Data Science...  negative   
2   Incoming Google DS-R interview — what to expect?   neutral   
3                         Data Science Online course   neutral   
4                          Maximum Likelihood Monday   neutral   
5   Incoming Google DS-R interview — what to expect?   neutral   
6   Why Is Ridge Giving Large Negative Coefficients?  negative   
7  A Library for Exploring Predictive Multiplicit...   neutral   
8  Backtesting an ensemble model's picks against ...   neutral   
9  Weekly Entering & Transitioning - Thread 05 Oc...   neutral   

   sentiment_score  
0         0.764242  
1         0.649543  
2         0.918867  
3         0.922719  
4         0.707494  
5         0.943661  
6         0.527360  
7         0.927111  
8         0.920560  
9         0.826192  


In [24]:
df.to_csv("../data/sentiment_results.csv", index=False)

In [25]:
print(
    df["sentiment"].value_counts()
    .reindex(["positive", "neutral", "negative"], fill_value=0)
)

sentiment
positive    0
neutral     8
negative    2
Name: count, dtype: int64


In [26]:
df[["title","sentiment","sentiment_score"]].sort_values(
    by = "sentiment_score",
    ascending = True
)

,title,sentiment,sentiment_score
6,Why Is Ridge Giving Large Negative Coefficients?,negative,0.527360
1,Please Please Please do not get a Data Science...,negative,0.649543
4,Maximum Likelihood Monday,neutral,0.707494
0,Model Selection Technique for Causal Inference,neutral,0.764242
9,Weekly Entering & Transitioning - Thread 05 Oc...,neutral,0.826192
2,Incoming Google DS-R interview — what to expect?,neutral,0.918867
8,Backtesting an ensemble model's picks against ...,neutral,0.920560
3,Data Science Online course,neutral,0.922719
7,A Library for Exploring Predictive Multiplicit...,neutral,0.927111
5,Incoming Google DS-R interview — what to expect?,neutral,0.943661


In [27]:

print(df.loc[6, "full_text"])
print(df.loc[1, "full_text"])

Why Is Ridge Giving Large Negative Coefficients? [removed]
Please Please Please do not get a Data Science degree, even if you want to be a Data Scientist


In [28]:
print(df["full_text"].str.contains(
    r"\[removed\]|\[deleted\]",
    case = False,
    na = False
).sum())

5


In [29]:
removed_posts = df[
    df["full_text"].str.contains(
        r"\[removed\]|\[deleted\]",
        case=False,
        na=False
    )
]

print(removed_posts[["title", "full_text", "sentiment"]])

                                               title  \
3                         Data Science Online course   
5   Incoming Google DS-R interview — what to expect?   
6   Why Is Ridge Giving Large Negative Coefficients?   
7  A Library for Exploring Predictive Multiplicit...   
8  Backtesting an ensemble model's picks against ...   

                                           full_text sentiment  
3               Data Science Online course [removed]   neutral  
5  Incoming Google DS-R interview — what to expec...   neutral  
6  Why Is Ridge Giving Large Negative Coefficient...  negative  
7  A Library for Exploring Predictive Multiplicit...   neutral  
8  Backtesting an ensemble model's picks against ...   neutral  


In [30]:
print(removed_posts[["title", "selftext"]])

                                               title   selftext
3                         Data Science Online course  [removed]
5   Incoming Google DS-R interview — what to expect?  [removed]
6   Why Is Ridge Giving Large Negative Coefficients?  [removed]
7  A Library for Exploring Predictive Multiplicit...  [removed]
8  Backtesting an ensemble model's picks against ...  [removed]


In [31]:
df["usable_text"] = (
    df["full_text"]
    .str.replace(r"\[removed\]|\[deleted\]", "", regex=True)
    .str.strip()
)

print("Posts with usable text:", (df["usable_text"] != "").sum())
print("Posts without usable text:", (df["usable_text"] == "").sum())
                     

Posts with usable text: 10
Posts without usable text: 0


In [32]:
print(df[["title", "full_text", "usable_text"]].head(10))

                                               title  \
0     Model Selection Technique for Causal Inference   
1  Please Please Please do not get a Data Science...   
2   Incoming Google DS-R interview — what to expect?   
3                         Data Science Online course   
4                          Maximum Likelihood Monday   
5   Incoming Google DS-R interview — what to expect?   
6   Why Is Ridge Giving Large Negative Coefficients?   
7  A Library for Exploring Predictive Multiplicit...   
8  Backtesting an ensemble model's picks against ...   
9  Weekly Entering & Transitioning - Thread 05 Oc...   

                                           full_text  \
0  Model Selection Technique for Causal Inference...   
1  Please Please Please do not get a Data Science...   
2   Incoming Google DS-R interview — what to expect?   
3               Data Science Online course [removed]   
4                          Maximum Likelihood Monday   
5  Incoming Google DS-R interview — what to exp

In [33]:
df["predictions"] = df["usable_text"].apply(
    predict_sentiment,
    args=(sentiment_pipeline,)
)

df["sentiment"] = df["predictions"].apply(lambda x: x["label"])
df["sentiment_score"] = df["predictions"].apply(lambda x: x["score"])

In [34]:
print(df["sentiment"].value_counts())

sentiment
neutral     8
negative    2
Name: count, dtype: int64


In [35]:
df.to_csv("../data/sentiment_results.csv", index=False)

In [36]:
print(df[["title", "usable_text", "sentiment"]].head(5).to_string(index=False))

                                                                                         title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [37]:
large_df = pd.read_csv("../data/processed_data_large.csv")
print(large_df.shape)


(100, 8)


In [38]:
print(large_df.columns.tolist())

['created_utc', 'id', 'num_comments', 'score', 'selftext', 'subreddit', 'title', 'full_text']


In [67]:
import sys
sys.path.append("..")

from ml.sentiment import load_sentiment_pipeline, predict_sentiment

sentiment_pipeline = load_sentiment_pipeline()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
large_df["predictions"] = large_df["full_text"].apply(predict_sentiment, args=(sentiment_pipeline,))

In [41]:
large_df["sentiment"] = large_df["predictions"].apply(lambda x: x["label"])

In [42]:
large_df["sentiment_score"] = large_df["predictions"].apply(lambda x: x["score"])

In [43]:
print(large_df["sentiment"].value_counts())

print("\nSentiment percentages:")
print(
    (large_df["sentiment"].value_counts(normalize=True) * 100).round(2)
)

sentiment
neutral     81
negative    11
positive     8
Name: count, dtype: int64

Sentiment percentages:
sentiment
neutral     81.0
negative    11.0
positive     8.0
Name: proportion, dtype: float64


In [44]:
print(large_df["sentiment_score"].describe())

count    100.000000
mean       0.780876
std        0.128699
min        0.480257
25%        0.703798
50%        0.808991
75%        0.881701
max        0.946157
Name: sentiment_score, dtype: float64


In [45]:
least_confident = large_df[
    ["title", "full_text", "sentiment", "sentiment_score"]
].sort_values(
    by="sentiment_score",
    ascending=True
).head(10)

print(
    least_confident[
        ["title", "sentiment", "sentiment_score"]
    ].to_string(index=False)
)

                                                                                                                                                                  title sentiment  sentiment_score
                                                                                         Those who have gotten a new job in the last 6 months - 1 year, how’s it going?   neutral         0.480257
Advice and professional development online feels like its geared towards one-time ad-hoc projects. Any resources on managing data on a recurring, organizational basis?  negative         0.488648
                                                         Is "end-to-end ownership" actually good for Data Scientists, or is it just unpaid glue work with a nicer name?   neutral         0.489886
                                                             Notes on a local NL2SQL assistant against real databases: schema selection, valid but wrong SQL, read-only  negative         0.513837
                         

In [46]:
for _, row in least_confident.head(3).iterrows():
    print("\n" + "=" * 80)
    print("TITLE:", row["title"])
    print("PREDICTED SENTIMENT:", row["sentiment"])
    print("CONFIDENCE:", round(row["sentiment_score"], 3))
    print("\nFULL TEXT:")
    print(row["full_text"][:1500])


TITLE: Those who have gotten a new job in the last 6 months - 1 year, how’s it going?
PREDICTED SENTIMENT: neutral
CONFIDENCE: 0.48

FULL TEXT:
Those who have gotten a new job in the last 6 months - 1 year, how’s it going? Getting a little bored in my current role where I’ve been for a few years (work at a large financial institution). Role feels very stable even with AI disruption, have access to many of the tools although for my specific job they’re not as helpful outside of research. I’m able to do all my work quickly and efficiently, don’t have any career growth opportunities as I’ve topped out the IC track here but my raises every year have beaten inflation as I’ve performed pretty well. Those who have changed roles, how has it gone for you? Interested in hearing from folks who’ve moved into the tech industry specifically. I figure it’ll take me a year+ to find a new job that I want more than my current one given the market and difficulty others have described.

TITLE: Advice and

In [50]:
print(predict_sentiment(None, sentiment_pipeline))

ValueError: You need to specify either `text` or `text_target`.

In [51]:
print(predict_sentiment("   ", sentiment_pipeline))

{'label': 'neutral', 'score': 0.7926775813102722}


In [ ]:
print(predict_sentiment(123, sentiment_pipeline))

ValueError: You need to specify either `text` or `text_target`.

In [52]:
print(predict_sentiment("I love machine learning!", sentiment_pipeline))

{'label': 'positive', 'score': 0.9869378805160522}


In [53]:
import inspect
print(inspect.getsource(predict_sentiment))

def predict_sentiment(text, sentiment_pipeline):

    # Step 1: Handle missing input
    if text is None:
        return {
            "label": None,
            "score": None,
            "error": "Empty text"
        }

    # Step 2: Convert non-string input to a string
    text = str(text)

    # Step 3: Handle empty text
    if text.strip() == "":
        return {
            "label": None,
            "score": None,
            "error": "Empty text"
        }

    # Step 4: Predict sentiment
    result = sentiment_pipeline(
        text,
        truncation=True,
        max_length=512
    )

    return result[0]



In [54]:
import importlib
import ml.sentiment

importlib.reload(ml.sentiment)

predict_sentiment = ml.sentiment.predict_sentiment

In [55]:
print(predict_sentiment(None, sentiment_pipeline))

{'label': None, 'score': None, 'error': 'Empty text'}


In [56]:
print(predict_sentiment(None, sentiment_pipeline))

print(predict_sentiment("   ", sentiment_pipeline))

print(predict_sentiment("I love machine learning!", sentiment_pipeline))

print(predict_sentiment(123, sentiment_pipeline))

{'label': None, 'score': None, 'error': 'Empty text'}
{'label': None, 'score': None, 'error': 'Empty text'}
{'label': 'positive', 'score': 0.9869378805160522}
{'label': 'neutral', 'score': 0.6865301728248596}


In [59]:
import pandas as pd

test_df = pd.DataFrame({
    "full_text": [
        "I love machine learning!",
        None,
        "   ",
        "This project is frustrating."
    ]
})

test_df["predictions"] = test_df["full_text"].apply(
    predict_sentiment,
    args=(sentiment_pipeline,)
)

test_df["sentiment"] = test_df["predictions"].apply(
    lambda x: x["label"]
)

test_df["sentiment_score"] = test_df["predictions"].apply(
    lambda x: x["score"]
)

print(test_df[["full_text", "sentiment", "sentiment_score"]])

                      full_text sentiment  sentiment_score
0      I love machine learning!  positive         0.986938
1                           NaN       NaN              NaN
2                                     NaN              NaN
3  This project is frustrating.  negative         0.942977


In [64]:
import importlib
import ml.sentiment

importlib.reload(ml.sentiment)

predict_sentiment = ml.sentiment.predict_sentiment

In [60]:
large_df = pd.read_csv("../data/processed_data_large.csv")

large_df["predictions"] = large_df["full_text"].apply(
    predict_sentiment,
    args=(sentiment_pipeline,)
)

large_df["sentiment"] = large_df["predictions"].apply(
    lambda x: x["label"]
)

large_df["sentiment_score"] = large_df["predictions"].apply(
    lambda x: x["score"]
)

large_df.to_csv("../data/sentiment_results.csv", index=False)

print("Dataset shape:", large_df.shape)
print("\nSentiment distribution:")
print(large_df["sentiment"].value_counts(dropna=False))

print("\nMissing sentiment labels:", large_df["sentiment"].isna().sum())

Dataset shape: (100, 11)

Sentiment distribution:
sentiment
neutral     81
negative    11
positive     8
Name: count, dtype: int64

Missing sentiment labels: 0


In [61]:
# Remove the intermediate predictions column
large_df = large_df.drop(columns=["predictions"])

# Save the cleaned sentiment results
large_df.to_csv("../data/sentiment_results.csv", index=False)

print("Saved successfully!")
print("Final dataset shape:", large_df.shape)
print(large_df[["title", "sentiment", "sentiment_score"]].head())

Saved successfully!
Final dataset shape: (100, 10)
                                               title sentiment  \
0  We ran some benchmarks comparing Jev against M...   neutral   
1  Most production “AI agents” are classifiers wi...   neutral   
2  Weekly Entering & Transitioning - Thread 12 Oc...   neutral   
3  4 Types of Case Studies in 2026 Data Science I...   neutral   
4   Amazon L5/L6 Data Science Coding Interview Prep?   neutral   

   sentiment_score  
0         0.916092  
1         0.656110  
2         0.816832  
3         0.905949  
4         0.938459  


In [66]:
print(analyze_sentiment("I love this project!", sentiment_pipeline))
print(analyze_sentiment(None, sentiment_pipeline))

{'sentiment': 'positive', 'confidence': 0.9849907755851746, 'error': None}
{'sentiment': None, 'confidence': None, 'error': 'Empty text'}


In [65]:
import importlib
import ml.sentiment

importlib.reload(ml.sentiment)

predict_sentiment = ml.sentiment.predict_sentiment
analyze_sentiment = ml.sentiment.analyze_sentiment

In [68]:
import importlib
import ml.sentiment

importlib.reload(ml.sentiment)

from ml.sentiment import (
    load_sentiment_pipeline,
    predict_sentiment,
    analyze_sentiment
)

print("All three functions imported successfully!")

All three functions imported successfully!


In [69]:
# Test 1: Positive text
print(analyze_sentiment("I love this project!", sentiment_pipeline))

# Test 2: Negative text
print(analyze_sentiment("This project is terrible.", sentiment_pipeline))

# Test 3: Missing input
print(analyze_sentiment(None, sentiment_pipeline))

# Test 4: Empty text
print(analyze_sentiment("   ", sentiment_pipeline))

{'sentiment': 'positive', 'confidence': 0.9849907755851746, 'error': None}
{'sentiment': 'negative', 'confidence': 0.9466939568519592, 'error': None}
{'sentiment': None, 'confidence': None, 'error': 'Empty text'}
{'sentiment': None, 'confidence': None, 'error': 'Empty text'}


In [70]:
import pandas as pd

results_df = pd.read_csv("../data/sentiment_results.csv")

print("Shape:", results_df.shape)
print("\nColumns:")
print(results_df.columns.tolist())

print("\nFirst five predictions:")
print(
    results_df[
        ["title", "sentiment", "sentiment_score"]
    ].head()
)

Shape: (100, 10)

Columns:
['created_utc', 'id', 'num_comments', 'score', 'selftext', 'subreddit', 'title', 'full_text', 'sentiment', 'sentiment_score']

First five predictions:
                                               title sentiment  \
0  We ran some benchmarks comparing Jev against M...   neutral   
1  Most production “AI agents” are classifiers wi...   neutral   
2  Weekly Entering & Transitioning - Thread 12 Oc...   neutral   
3  4 Types of Case Studies in 2026 Data Science I...   neutral   
4   Amazon L5/L6 Data Science Coding Interview Prep?   neutral   

   sentiment_score  
0         0.916092  
1         0.656110  
2         0.816832  
3         0.905949  
4         0.938459  
